In [2]:
import os
import glob
import h5py
import torch
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import plotly.express as px
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
parent_dir = r"W:\pathologie\bioinfo-archive\UK_Augsburg_Claus\KristianUnger\TitanFeatures\20x_512px_0px_overlap\slide_features_titan"
h5_key = "features"
df_path = "dataframes/pediatric_imputed_survival.csv"
df = pd.read_csv(df_path)
df = df.dropna(subset=['scan_name']).reset_index(drop=True)
df

,position,sample_name,slide_name,scan_name,tif_name,slide_id,exists,sample_name_alt,sample_type,sample_type_alt,...,Art_event,type_event,EFS_years,EFS_days,Death_disease,Date_of_death,OS_days,LastFollowup,date_of_LFU,Methylierungsgruppe
0,A2,1-T,T/1999/149,slide-2025-10-29T08-13-45-R1-S1.mrxs,slide-2025-10-29T08-13-45-R1-S1.ome.tif,slide-2025-10-29T08-13-45-R1-S1,slide-2025-10-29T08-13-45-R1-S1,1T,tumor,tumor,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,2.0,2009-08-20 00:00:00,3
1,A3,2-b-T,T/2011/1450-b,slide-2025-10-29T08-25-47-R1-S2.mrxs,slide-2025-10-29T08-25-47-R1-S2.ome.tif,slide-2025-10-29T08-25-47-R1-S2,slide-2025-10-29T08-25-47-R1-S2,2bT,tumor,tumor,...,9,NaN,NaN,3162.0,1.0,NaN,3162.0,1.0,NaN,x
2,A6,2-d-T,T/2011/1450-d,slide-2025-10-29T08-35-06-R1-S3.mrxs,slide-2025-10-29T08-35-06-R1-S3.ome.tif,slide-2025-10-29T08-35-06-R1-S3,slide-2025-10-29T08-35-06-R1-S3,2dT,tumor,tumor,...,NaN,NaN,NaN,2318.0,1.0,NaN,2318.0,NaN,NaN,NaN
3,A8,3-T,T/2013/785,slide-2025-10-29T08-54-08-R1-S5.mrxs,slide-2025-10-29T08-54-08-R1-S5.ome.tif,slide-2025-10-29T08-54-08-R1-S5,slide-2025-10-29T08-54-08-R1-S5,3T,tumor,tumor,...,9,NaN,NaN,2555.0,1.0,NaN,2555.0,1.0,NaN,x
4,A9,4-T,T/2013/569,slide-2025-10-29T09-04-19-R1-S6.mrxs,slide-2025-10-29T09-04-19-R1-S6.ome.tif,slide-2025-10-29T09-04-19-R1-S6,slide-2025-10-29T09-04-19-R1-S6,4T,tumor,tumor,...,1.3. weitere Events . 25.02.14. 28.05.14,4.0,1.010959e+09,369.0,2.0,2015-07-01 00:00:00,901.0,2.0,NaN,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,G4,75-T,KT2008/1381,slide-2025-10-30T15-59-21-R2-S8.mrxs,slide-2025-10-30T15-59-21-R2-S8.ome.tif,slide-2025-10-30T15-59-21-R2-S8,slide-2025-10-30T15-59-21-R2-S8,75T,tumor,tumor,...,9,NaN,NaN,4651.0,1.0,NaN,4651.0,1.0,NaN,x
76,G5,76-T,KT/2008/1362-1,slide-2025-10-30T16-08-37-R2-S9.mrxs,slide-2025-10-30T16-08-37-R2-S9.ome.tif,slide-2025-10-30T16-08-37-R2-S9,slide-2025-10-30T16-08-37-R2-S9,76T,tumor,tumor,...,01/2009 1.Lokalrezidiv links mit bipulmonale M...,3.0,3.835616e-01,140.0,2.0,2010-08-28 00:00:00,729.0,2.0,2010-08-28 00:00:00,x
77,G6,77-T,KT/2008/1365,slide-2025-10-30T16-26-23-R2-S10.mrxs,slide-2025-10-30T16-26-23-R2-S10.ome.tif,slide-2025-10-30T16-26-23-R2-S10,slide-2025-10-30T16-26-23-R2-S10,77T,tumor,tumor,...,9,NaN,NaN,5121.0,1.0,NaN,5121.0,1.0,NaN,1
78,G8,78-T,KT/2008/956-1,slide-2025-10-30T16-36-42-R2-S11.mrxs,slide-2025-10-30T16-36-42-R2-S11.ome.tif,slide-2025-10-30T16-36-42-R2-S11,slide-2025-10-30T16-36-42-R2-S11,78T,tumor,tumor,...,9,NaN,NaN,NaN,1.0,,NaN,1.0,NaN,3


In [4]:
embeddings = []
valid_indices = []
scan_names = []
pat_ids = []

print("Extracting embeddings from .h5 files...")
for idx, row in df.iterrows():
    try:
        scan_name = row['scan_name'][:-5]  # remove .mrxs extension
    except Exception as e:
        print(e)
    pat_id = str(row['ID_patient']) + scan_name.split("-")[-1]

    file_name = f"{scan_name}.h5" if not scan_name.endswith('.h5') else scan_name
    h5_path = os.path.join(parent_dir, file_name)

    # Check if file exists to prevent pipeline crashes
    if os.path.exists(h5_path):
        try:
            with h5py.File(h5_path, 'r') as f:
                vector = f[h5_key][:]
                # Flatten to 1D if necessary
                if vector.ndim > 1:
                    vector = np.squeeze(vector)

                if scan_name not in scan_names:
                    embeddings.append(vector)
                    valid_indices.append(idx)
                    scan_names.append(scan_name)
                    pat_ids.append(pat_id)
                else:
                    print(f"repeated scan: {scan_name}")

        except Exception as e:
            print(f"Error reading {file_name}: {e}")
    else:
        print(f"Warning: Missing file for {scan_name}. Skipping.")

Extracting embeddings from .h5 files...


In [5]:
def main():
    # ==========================================
    # 1. Configuration & Paths
    # =========================================


    X = np.array(embeddings)

    # ==========================================
    # 3. The Elbow Method (Calculate Inertia)
    # ==========================================
    max_k = min(10, len(X)) # Don't test more clusters than we have slides!
    k_values = range(1, max_k + 1)
    inertias = []

    print(f"Testing k=1 to k={max_k}...")

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
        kmeans.fit(X)
        inertias.append(kmeans.inertia_) # Inertia = Within-Cluster-Sum-of-Squares (WCSS)

    # ==========================================
    # 4. Plot the Elbow Curve
    # ==========================================
    print("Generating Elbow plot...")

    df_elbow = pd.DataFrame({
        'Number of Clusters (k)': k_values,
        'Inertia (WCSS)': inertias
    })

    fig = px.line(
        df_elbow,
        x='Number of Clusters (k)',
        y='Inertia (WCSS)',
        markers=True,
        title="Elbow Method for Optimal k (WSI Embeddings)",
        template="plotly_white"
    )

    # Improve aesthetics to make the "elbow" easier to spot
    fig.update_traces(marker=dict(size=10, color='royalblue', line=dict(width=2, color='DarkSlateGrey')), line=dict(width=3))
    fig.update_layout(xaxis=dict(tickmode='linear', tick0=1, dtick=1)) # Force x-axis to show every integer

    # Make sure to write to HTML if the VS Code terminal/Jupyter setup is still giving you trouble!
    # fig.write_html("elbow_plot.html", auto_open=True)
    fig.show()

if __name__ == "__main__":
    main()

Testing k=1 to k=10...


C:\Users\gomaaad\Projects\nsclc_gomaaad\.conda\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\gomaaad\Projects\nsclc_gomaaad\.conda\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\gomaaad\Projects\nsclc_gomaaad\.conda\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\gomaaad\Projects\nsclc_gomaaad\.conda\lib\site-packages\sklea

Generating Elbow plot...


In [10]:
def main():

    slide_ids = scan_names

    X = np.array(embeddings)

    # ==========================================
    # 3. Discover Phenotypes (Clustering)
    # ==========================================
    num_clusters = 3 # Adjust this based on how many groups you suspect exist
    print(f"Running K-Means clustering to find {num_clusters} groups...")

    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init="auto")
    cluster_assignments = kmeans.fit_predict(X)
    cluster_labels = [f"Cluster {c}" for c in cluster_assignments]

    # Add cluster column to df
    df['cluster'] = np.nan
    df.loc[valid_indices, 'cluster'] = cluster_assignments

    # One-hot encoding
    df['cluster_0'] = (df['cluster'] == 0).astype(int)
    df['cluster_1'] = (df['cluster'] == 1).astype(int)
    df['cluster_2'] = (df['cluster'] == 2).astype(int)

    # ==========================================
    # 4. Compute t-SNE (Only Once!)
    # ==========================================
    print(f"Computing t-SNE for {X.shape[0]} slides...")
    perplexity_value = min(30, max(5, len(X) - 1))

    tsne = TSNE(n_components=2, perplexity=perplexity_value, random_state=42, init='pca')
    X_tsne = tsne.fit_transform(X)

    # Store everything in a DataFrame for easy plotting
    df_plot = pd.DataFrame({
        'TSNE_1': X_tsne[:, 0],
        'TSNE_2': X_tsne[:, 1],
        'Slide_ID': slide_ids,
        'Cluster': cluster_labels
    })

    # ==========================================
    # 5. Create Side-by-Side Interactive Plot
    # ==========================================
    print("Generating side-by-side plot...")

    # Initialize a 1x2 grid of subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Before Clustering (Raw Distribution)", "After Clustering (K-Means)")
    )

    # --- LEFT PLOT: Before Clustering ---
    fig.add_trace(
        go.Scatter(
            x=df_plot['TSNE_1'],
            y=df_plot['TSNE_2'],
            mode='markers',
            marker=dict(color='lightgrey', size=8, opacity=0.8, line=dict(width=1, color='gray')),
            text=df_plot['Slide_ID'], # Shows Slide ID on hover
            hoverinfo='text',
            name="Unlabeled Slides",
            showlegend=False
        ),
        row=1, col=1
    )

    # --- RIGHT PLOT: After Clustering ---
    # We loop through each cluster to add them separately so they show up nicely in the legend
    colors = px.colors.qualitative.Bold
    sorted_clusters = sorted(df_plot['Cluster'].unique())

    for i, cluster_name in enumerate(sorted_clusters):
        df_sub = df_plot[df_plot['Cluster'] == cluster_name]

        fig.add_trace(
            go.Scatter(
                x=df_sub['TSNE_1'],
                y=df_sub['TSNE_2'],
                mode='markers',
                marker=dict(color=colors[i % len(colors)], size=8, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')),
                text=df_sub['Slide_ID'],
                hoverinfo='text',
                name=cluster_name
            ),
            row=1, col=2
        )

    # Update layout for a clean look
    fig.update_layout(
        title_text="WSI Foundation Model Embeddings Analysis",
        template="plotly_white",
        height=600, # Make the plot a bit taller for better visibility
        hovermode="closest"
    )

    # Hide the axes coordinates for a cleaner visualization
    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
    fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)

    fig.show()

    # Save the updated dataframe to CSV
    df.to_csv(r'pediatric_imputed_survival_mit_clusters.csv', index=False)

    # Uncomment the line below to save a permanent interactive HTML file
    # fig.write_html("tsne_before_and_after.html")

if __name__ == "__main__":
    main()

Running K-Means clustering to find 3 groups...
Computing t-SNE for 80 slides...


C:\Users\gomaaad\Projects\nsclc_gomaaad\.conda\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Generating side-by-side plot...


In [11]:
print(pd.crosstab(df['cluster'], df['Death_disease']))

Death_disease  1.0  2.0
cluster                
0.0             34    1
1.0             22    8
2.0             11    4


In [7]:
import umap
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def main():

    slide_ids = scan_names
    X = np.array(embeddings)

    # ==========================================
    # 3. Discover Phenotypes (Clustering)
    # ==========================================
    num_clusters = 3
    print(f"Running K-Means clustering to find {num_clusters} groups...")

    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init="auto")
    cluster_assignments = kmeans.fit_predict(X)
    cluster_labels = [f"Cluster {c}" for c in cluster_assignments]

    # ==========================================
    # 4. Compute UMAP (Only Once!)
    # ==========================================
    print(f"Computing UMAP for {X.shape[0]} slides...")

    # n_neighbors controls how UMAP balances local versus global structure.
    # Default is 15. It must be strictly less than the number of samples.
    n_neighbors_value = min(15, max(2, len(X) - 1))

    reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors_value, random_state=42)
    X_umap = reducer.fit_transform(X)

    # Store everything in a DataFrame for easy plotting
    df_plot = pd.DataFrame({
        'UMAP_1': X_umap[:, 0],
        'UMAP_2': X_umap[:, 1],
        'Slide_ID': slide_ids,
        'Cluster': cluster_labels
    })

    # ==========================================
    # 5. Create Side-by-Side Interactive Plot
    # ==========================================
    print("Generating side-by-side plot...")

    # Initialize a 1x2 grid of subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Before Clustering (Raw Distribution)", "After Clustering (K-Means)")
    )

    # --- LEFT PLOT: Before Clustering ---
    fig.add_trace(
        go.Scatter(
            x=df_plot['UMAP_1'],
            y=df_plot['UMAP_2'],
            mode='markers',
            marker=dict(color='lightgrey', size=8, opacity=0.8, line=dict(width=1, color='gray')),
            text=df_plot['Slide_ID'],
            hoverinfo='text',
            name="Unlabeled Slides",
            showlegend=False
        ),
        row=1, col=1
    )

    # --- RIGHT PLOT: After Clustering ---
    colors = px.colors.qualitative.Bold
    sorted_clusters = sorted(df_plot['Cluster'].unique())

    for i, cluster_name in enumerate(sorted_clusters):
        df_sub = df_plot[df_plot['Cluster'] == cluster_name]

        fig.add_trace(
            go.Scatter(
                x=df_sub['UMAP_1'],
                y=df_sub['UMAP_2'],
                mode='markers',
                marker=dict(color=colors[i % len(colors)], size=8, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')),
                text=df_sub['Slide_ID'],
                hoverinfo='text',
                name=cluster_name
            ),
            row=1, col=2
        )

    # Update layout for a clean look
    fig.update_layout(
        title_text="WSI Foundation Model Embeddings Analysis (UMAP)",
        template="plotly_white",
        height=600,
        hovermode="closest"
    )

    # Hide the axes coordinates for a cleaner visualization
    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
    fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)

    fig.show()

    # fig.write_html("umap_before_and_after.html")

if __name__ == "__main__":
    main()

Running K-Means clustering to find 3 groups...
Computing UMAP for 84 slides...


C:\Users\gomaaad\Projects\nsclc_gomaaad\.conda\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\gomaaad\Projects\nsclc_gomaaad\.conda\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Generating side-by-side plot...
